#### 0. Инициализация

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt

from dsp_basic import fft_dit
from filter_MA import design_ma, apply_ma
from filter_FIR import design_fir_hp_rect, apply_fir
from filter_IIR import design_iir_hp_onepole, apply_iir

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['grid.alpha'] = 0.3

#### 1. Параметры сигнала и генерация

In [ ]:
# Параметры дискретизации 
fs = 400          # частота дискретизации, Гц
duration = 1.0    # длительность, c

# Параметры исходного сигнала
A1 = 1.0          # амплитуда НЧ-синусоиды
f1 = 5.0          # частота НЧ-синусоиды, Гц

A2 = 0.5          # амплитуда ВЧ-синусоиды
f2 = 60.0         # частота ВЧ-синусоиды, Гц

t = np.arange(0, duration, 1/fs)

x = A1 * np.sin(2 * np.pi * f1 * t) + A2 * np.sin(2 * np.pi * f2 * t)

plt.plot(t, x, label='x(t) исходный')
plt.xlabel('t, c')
plt.ylabel('Амплитуда')
plt.title('Исходный комбинированный сигнал x(t)')
plt.grid(True)
plt.legend()
plt.show()

#### 2. Параметры однородного, КИХ и БИХ фильтров

In [ ]:
# 2. Параметры фильтров

# Однородный (МА)
ma_length = 21      # длина окна N

# FIR ВЧ (прямоугольное окно)
fc_hp_fir = 20.0    # частота среза, Гц
fir_length = 101    # длина фильтра

# IIR ВЧ (однополюсный)
fc_hp_iir = 20.0    # частота среза, Гц


#### 3. Расчет коэффициентов фильтров

In [ ]:
# 3.1. Однородный фильтр (MA)
h_ma = design_ma(ma_length)
print("MA h[n] =", h_ma)

# 3.2. КИХ ВЧ фильтр (FIR) с прямоугольным окном
h_fir = design_fir_hp_rect(
    cutoff_hz=fc_hp_fir,
    sample_rate_hz=fs,
    filter_length=fir_length
)
print("FIR HP h[n] =", h_fir)

# 3.3. БИХ ВЧ фильтр (IIR) с одним полюсом
a_iir, b_iir = design_iir_hp_onepole(
    cutoff_hz=fc_hp_iir,
    sample_rate_hz=fs
)
print("IIR HP a =", a_iir)
print("IIR HP b =", b_iir)

#### 4. Формирование искажённого сигнала

In [ ]:
# НЧ-помеха
A_noise = 2.0
w_noise = 0.5 # медленная синусоида

noise = A_noise * np.sin(w_noise * t)
x_distorted = x + noise

plt.plot(t, x, label='Исходный', alpha=0.7)
plt.plot(t, x_distorted, label='Искажённый', alpha=0.7)
plt.xlabel('t, c')
plt.ylabel('Амплитуда')
plt.title('Сигнал до и после искажения')
plt.grid(True)
plt.legend()
plt.show()


#### 5. Обработка искажённого сигнала фильтрами

In [ ]:
# apply_* ожидают Python-списки → конвертируем

x_list = x_distorted.tolist()

y_ma  = apply_ma(x_list, h_ma)
y_fir = apply_fir(x_list, h_fir)
y_iir = apply_iir(x_list, a_iir, b_iir)

y_ma  = np.array(y_ma)
y_fir = np.array(y_fir)
y_iir = np.array(y_iir)

# Графики

plt.plot(t, x_distorted, label='Искажённый', alpha=0.5)
plt.plot(t, y_ma, label='После MA', alpha=0.8)
plt.xlabel('t, c')
plt.ylabel('Амплитуда')
plt.title('Однородный фильтр (МА)')
plt.grid(True)
plt.legend()
plt.show()

plt.plot(t, x_distorted, label='Искажённый', alpha=0.5)
plt.plot(t, y_fir, label='После FIR HP', alpha=0.8)
plt.xlabel('t, c')
plt.ylabel('Амплитуда')
plt.title('КИХ ВЧ‑фильтр (FIR)')
plt.grid(True)
plt.legend()
plt.show()

plt.plot(t, x_distorted, label='Искажённый', alpha=0.5)
plt.plot(t, y_iir, label='После IIR HP', alpha=0.8)
plt.xlabel('t, c')
plt.ylabel('Амплитуда')
plt.title('БИХ ВЧ‑фильтр (IIR)')
plt.grid(True)
plt.legend()
plt.show()

#### 6. Амплитудно-частотные характеристики фильтров

In [ ]:
N_fft = 1024
imp = np.zeros(N_fft)
imp[0] = 1.0

# MA
h_ma_imp = apply_ma(imp.tolist(), h_ma)
h_ma_imp = np.array(h_ma_imp[:N_fft])
H_ma = np.fft.fft(h_ma_imp, n=N_fft)

# FIR
h_fir_imp = apply_fir(imp.tolist(), h_fir)
h_fir_imp = np.array(h_fir_imp[:N_fft])
H_fir = np.fft.fft(h_fir_imp, n=N_fft)

# IIR
h_iir_imp = apply_iir(imp.tolist(), a_iir, b_iir)
h_iir_imp = np.array(h_iir_imp[:N_fft])
H_iir = np.fft.fft(h_iir_imp, n=N_fft)

freqs = np.linspace(0, fs/2, N_fft//2, endpoint=False)

H_ma_db  = 20*np.log10(np.abs(H_ma[:N_fft//2])  + 1e-12)
H_fir_db = 20*np.log10(np.abs(H_fir[:N_fft//2]) + 1e-12)
H_iir_db = 20*np.log10(np.abs(H_iir[:N_fft//2]) + 1e-12)

plt.plot(freqs, H_ma_db, label='MA')
plt.plot(freqs, H_fir_db, label='FIR HP')
plt.plot(freqs, H_iir_db, label='IIR HP')
plt.xlabel('f, Гц')
plt.ylabel('Амплитуда, дБ')
plt.title('АЧХ фильтров')
plt.grid(True)
plt.legend()
plt.show()